In [1]:
from pathlib import Path
import cv2
import numpy as np

root = Path("/work/tesi_cmininno/dataset/CholecSeg8k")

image_path = next(root.glob("video*/video*/*_endo.png"))
mask_path = Path(
    str(image_path).replace(
        "_endo.png",
        "_endo_watershed_mask.png"
    )
)

print("Image:", image_path)
print("Mask :", mask_path)

image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

print("\nIMAGE")
print("shape:", image.shape)
print("dtype:", image.dtype)

print("\nMASK")
print("shape:", mask.shape)
print("dtype:", mask.dtype)
print("unique:", np.unique(mask))

Image: /work/tesi_cmininno/dataset/CholecSeg8k/video55/video55_00508/frame_586_endo.png
Mask : /work/tesi_cmininno/dataset/CholecSeg8k/video55/video55_00508/frame_586_endo_watershed_mask.png

IMAGE
shape: (480, 854, 3)
dtype: uint8

MASK
shape: (480, 854, 3)
dtype: uint8
unique: [ 11  12  13  21  22  31  32  50 255]


In [2]:
if mask.ndim == 3:
    print("Mask is RGB/BGR")
    for c in range(mask.shape[2]):
        print(f"Channel {c}:", np.unique(mask[:, :, c])[:50])
# else:
    print("Mask is grayscale")

Mask is RGB/BGR
Channel 0: [ 11  12  13  21  22  31  32  50 255]
Channel 1: [ 11  12  13  21  22  31  32  50 255]
Channel 2: [ 11  12  13  21  22  31  32  50 255]
Mask is grayscale


In [3]:
print("Image size:", image.shape[:2])
print("Mask size :", mask.shape[:2])

Image size: (480, 854)
Mask size : (480, 854)


In [4]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("../")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.cholecseg8k import CholecSeg8kDataset

In [5]:
DATASET_ROOT = "/work/tesi_cmininno/dataset/CholecSeg8k"

print(Path(DATASET_ROOT).exists())
print(Path(DATASET_ROOT))

True
/work/tesi_cmininno/dataset/CholecSeg8k


In [6]:
root = Path(DATASET_ROOT)

video_dirs = sorted(
    p for p in root.iterdir()
    if p.is_dir() and p.name.startswith("video")
)

print("Number of videos:", len(video_dirs))

for video_dir in video_dirs:
    print(video_dir.name)

Number of videos: 17
video01
video09
video12
video17
video18
video20
video24
video25
video26
video27
video28
video35
video37
video43
video48
video52
video55


In [7]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

SEED = 42

In [8]:
dataset = CholecSeg8kDataset(
    root=DATASET_ROOT
)

print("Total samples:", len(dataset))
print("Videos:", dataset.video_ids)

Total samples: 8080
Videos: [1, 9, 12, 17, 18, 20, 24, 25, 26, 27, 28, 35, 37, 43, 48, 52, 55]


In [9]:
train, val, test = dataset.create_split(
    strategy="percentage",
    train_ratio=0.70,
    val_ratio=0.30,
    test_ratio=0,
    seed=42,
    stratified=True,
)


DATASET SPLIT

Train:
  Groups : [1, 9, 12, 20, 26, 27, 28, 35, 37, 52, 55]
  Frames : 5360

Validation:
  Groups : [17, 18, 24, 25, 43, 48]
  Frames : 2720

Total:
  Groups : 17
  Frames : 8080


In [ ]:
image = sample["image"]
mask = sample["mask"]

print("Image:")
print("  shape:", image.shape)
print("  dtype:", image.dtype)

print("\nMask:")
print("  shape:", mask.shape)
print("  dtype:", mask.dtype)

print("\nMetadata:")
print("  video:", sample["video"])
print("  clip :", sample["clip"])
print("  frame:", sample["frame"])

In [ ]:
unique_classes = np.unique(mask)

print("Classes present:", unique_classes)
print("Number of classes:", len(unique_classes))

In [ ]:
print("Original -> semantic class")

for original_value, class_id in sorted(
    dataset.LABEL_MAP.items()
):
    print(
        f"{original_value:>3} -> "
        f"{class_id:>2} "
        f"({dataset.CLASS_NAMES[class_id]})"
    )

In [ ]:
from src.metrics.metric_manager import MetricManager
import monai.metrics

In [ ]:
metrics_dict = {
    "dice": monai.metrics.DiceMetric(
        include_background=True, 
        reduction="mean"
    ),
    "iou": monai.metrics.MeanIoU(
        include_background=True, 
        reduction="mean"
    ),
    "hd95": monai.metrics.HausdorffDistanceMetric(
        include_background=True, 
        percentile=95, 
        reduction="mean"
    )
}

In [ ]:
class_names = {
    0: "background",
    1: "abdominal_wall",
    2: "liver",
    3: "gastrointestinal_tract",
    4: "fat",
    5: "grasper",
    6: "connective_tissue",
    7: "blood",
    8: "cystic_duct",
    9: "l_hook_electrocautery",
    10: "gallbladder",
    11: "hepatic_vein",
    12: "liver_ligament"
}

In [10]:


metric_manager = MetricManager(
    metrics=metrics_dict,
    num_classes=13,
    dataset_name="cholecseg8k",
    class_names=class_names
)

NameError: name 'MetricManager' is not defined

In [ ]:
metric_manager.class_names

In [ ]:
plt.figure(figsize=(16, 5))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title(
    f"Image - {sample['video']} / {sample['frame']}"
)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="tab20")
plt.title("Semantic mask")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.imshow(image)
plt.imshow(
    np.ma.masked_where(mask == 0, mask),
    cmap="tab20",
    alpha=0.45,
)

plt.title(
    f"Overlay - {sample['video']} / {sample['frame']}"
)

plt.axis("off")
plt.show()

In [ ]:
present_classes = np.unique(mask)

print("Classes present in this frame:")

for class_id in present_classes:
    print(
        f"{class_id:2d} -> "
        f"{dataset.CLASS_NAMES[class_id]}"
    )

In [ ]:
def show_sample(dataset, index):

    sample = dataset[index]

    image = sample["image"]
    mask = sample["mask"]

    present_classes = np.unique(mask)

    print(
        f"Index: {index}\n"
        f"Video: {sample['video']}\n"
        f"Clip : {sample['clip']}\n"
        f"Frame: {sample['frame']}\n"
        f"Image: {image.shape}, {image.dtype}\n"
        f"Mask : {mask.shape}, {mask.dtype}\n"
    )

    print("Classes:")

    for class_id in present_classes:
        print(
            f"  {class_id:2d}: "
            f"{dataset.CLASS_NAMES[class_id]}"
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].imshow(image)
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="tab20")
    axes[1].set_title("Mask")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(
        np.ma.masked_where(mask == 0, mask),
        cmap="tab20",
        alpha=0.45,
    )
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_sample(dataset, 50)

In [ ]:
show_sample(dataset, 7000)

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
)

In [ ]:
batch = next(iter(loader))

print(batch.keys())